# Vendor Extraction Benchmark

This notebook compares local Ollama vendor extraction with batch sizes of 1, 10, and 20.

## Results

Batch size 10 is the best choice: its vendor results are more reliable than batch size 20 while processing each transaction much faster than one-at-a-time requests. Batch size 20 brings only a marginal additional speed gain, so it is not worth the reduction in output quality.

## 0. Imports

In [ ]:
# Step 0: Imports
import json
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display


## 1. Load transactions

In [ ]:
def load_transactions_csv(csv_path):
    try:
        return pd.read_csv(
            csv_path, sep=";", encoding="utf-8-sig", dtype="string", keep_default_na=False
        )
    except (FileNotFoundError, UnicodeDecodeError, pd.errors.ParserError) as error:
        raise ValueError(f"Could not load CSV: {error}") from error


data_folder = Path("../data")
transactions = pd.DataFrame()

for csv_path in sorted(data_folder.glob("*.csv")):
    transactions = pd.concat([transactions, load_transactions_csv(csv_path)], ignore_index=True)


## 2. Run benchmark

Each approach classifies the same first 20 transactions.

In [ ]:
OLLAMA_MODEL = "gemma3:4b"
OLLAMA_URL = "http://localhost:11434/api/generate"


def extract_vendors_batch(batch):
    batch_description = "\n\n".join(
        f"Transaction {number}:\n"
        + "\n".join(
            f"{column}: {transaction[column]}"
            for column in ["Beschreibung1", "Beschreibung2", "Beschreibung3"]
            if transaction[column]
        )
        for number, (_, transaction) in enumerate(batch.iterrows(), start=1)
    )

    prompt = (
        "Extract one vendor for each numbered bank transaction. Keep the vendor spelling "
        "from the description. For interest credits or service fees, the bank can be the vendor. "
        "Do not assign categories. Return JSON only, for example: "
        '{\"vendors\": {\"1\": \"Vendor A\", \"2\": \"Vendor B\"}}.\n\n'
        + batch_description
    )

    try:
        response = requests.post(
            OLLAMA_URL,
            json={"model": OLLAMA_MODEL, "prompt": prompt, "format": "json", "stream": False},
            timeout=60,
        )
        response.raise_for_status()
        vendors = json.loads(response.json()["response"])["vendors"]
        return [vendors[str(number)].strip() for number in range(1, len(batch) + 1)]
    except (requests.RequestException, json.JSONDecodeError, KeyError, AttributeError) as error:
        raise ValueError(f"Could not extract vendors: {error}") from error


batch_sizes = [1, 10, 20]
test_transactions = transactions.head(20).copy()
performance_rows = []
review_table = test_transactions[["Beschreibung1", "Beschreibung2", "Beschreibung3"]].copy()

for batch_size in batch_sizes:
    vendors = []
    request_count = 0
    start = time.perf_counter()

    for start_index in range(0, len(test_transactions), batch_size):
        batch = test_transactions.iloc[start_index : start_index + batch_size]
        vendors.extend(extract_vendors_batch(batch))
        request_count += 1

    performance_rows.append(
        {"Batch size": batch_size, "Requests": request_count, "Seconds": time.perf_counter() - start}
    )
    review_table[f"Vendor (batch {batch_size})"] = vendors

performance = pd.DataFrame(performance_rows)
performance["Seconds per transaction"] = performance["Seconds"] / len(test_transactions)
performance["Speedup vs. batch size 1"] = (
    performance.loc[0, "Seconds per transaction"] / performance["Seconds per transaction"]
)

display(performance)
display(review_table)
